### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item_extension (optional)

### Strategy:
- Extract conditions/diagnoses from dbo_vendor_item filtered by ItemType
- Map SnomedCode to OMOP concept_id via vocabulary tables
- Map StatusQualifier to condition_status_concept_id via domain_source_to_concept
- Calculate condition_end_date from Duration + DurationUnit when available
- Use condition_type_concept_id = 32817 (EHR encounter record)

### Notes:
- This notebook depends on source_to_person being populated
- provider_id and visit_occurrence_id will be NULL (mapping tables not yet created)
- SNOMED concept mapping requires OMOP vocabulary tables to be loaded
- condition_source_concept_id uses 0 if SNOMED code not found in vocabulary

In [0]:
source = 'allscripts_tw'

# Transformation

In [0]:
%sql
DESCRIBE TABLE _exponent.omop.condition_occurrence

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW standard_concept_mapping AS
SELECT
  concept.vocabulary_id           AS source_vocabulary_id,    -- ICD9CM / ICD10CM / SNOMED
  concept.concept_code            AS source_concept_code,
  concept.concept_id              AS source_concept_id,
  standard_concept.concept_id     AS standard_concept_id,
  standard_concept.concept_name   AS standard_concept_name
FROM _exponent.omop.concept
JOIN _exponent.omop.concept_relationship
  ON concept_relationship.concept_id_1 = concept.concept_id
 AND concept_relationship.relationship_id = 'Maps to'
JOIN _exponent.omop.concept AS standard_concept
  ON standard_concept.concept_id = concept_relationship.concept_id_2
 AND standard_concept.standard_concept = 'S'
 AND standard_concept.domain_id = 'Condition'
WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW problem_diagnosis AS
WITH problems AS (
SELECT 
CAST(id AS BIGINT) AS id,
entryname,
NULLIF(TRIM(REGEXP_REPLACE(Snomed3CODE,'[\\s\\u00A0]+', '')), '') AS SnomedCode,
NULLIF(TRIM(REGEXP_REPLACE(ICD10DiagnosisCode,'[\\s\\u00A0]+', '')), '') AS ICD10DiagnosisCode,
NULLIF(TRIM(REGEXP_REPLACE(ICD9DiagnosisCODE,'[\\s\\u00A0]+', '')), '') AS ICD9DiagnosisCODE
FROM
_exponent._bronze_allscripts_tw_works_vw.dbo_problem_de)
,problem_diagnosis AS (
  SELECT 
  id,
  entryname,
  COALESCE(SnomedCode, ICD10DiagnosisCode, ICD9DiagnosisCODE) AS DiagnosisCode,
    CASE 
    WHEN SnomedCode IS NOT NULL THEN 'SNOMED' 
    WHEN ICD10DiagnosisCode IS NOT NULL THEN 'ICD10CM' 
    WHEN ICD9DiagnosisCODE IS NOT NULL THEN 'ICD9CM' 
  END AS DiagnosisCodeType
FROM problems
)
SELECT 
problem_diagnosis.*,
standard_concept_mapping.*
FROM problem_diagnosis
LEFT JOIN standard_concept_mapping
  ON standard_concept_mapping.source_vocabulary_id = problem_diagnosis.DiagnosisCodeType
 AND standard_concept_mapping.source_concept_code = problem_diagnosis.DiagnosisCode


In [0]:
%sql
SELECT * FROM problem_diagnosis


In [0]:
%sql
SELECT
  CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_encounter_diagnosis',
       'id',
       CAST(dbo_encounter_diagnosis.id AS BIGINT)
  ) AS source_condition_occurrence_id, -- staging key
  source_to_person.person_id,
  COALESCE(
    standard_concept_mapping.standard_concept_id,
    problem_diagnosis.standard_concept_id,
    0
  ) AS condition_concept_id,
  CAST(dbo_encounter.dttm AS DATE)      AS condition_start_date,
  CAST(dbo_encounter.dttm AS TIMESTAMP) AS condition_start_datetime,
  NULL AS condition_end_date,
  NULL AS condition_end_datetime,
  CASE
    WHEN dbo_encounter_diagnosis.DiagnosisType IN ('ICD9','ICD10') THEN 32020
    WHEN dbo_encounter_diagnosis.DiagnosisType = 'PROBLEM' THEN 38000245
    ELSE 32817
  END AS condition_type_concept_id,
  NULL AS condition_status_concept_id,
  NULL AS stop_reason,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  COALESCE(
    NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD9_Diagnosis_DE.EntryCode,  '[\\s\\u00A0]+', '')), ''),
    NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD10_Diagnosis_DE.EntryCode, '[\\s\\u00A0]+', '')), ''),
    problem_diagnosis.DiagnosisCode
  ) AS condition_source_value,
  COALESCE(
    standard_concept_mapping.source_concept_id,
    problem_diagnosis.source_concept_id,
    0
  ) AS condition_source_concept_id,

  NULL AS condition_status_source_value

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_encounter_diagnosis
  ON dbo_encounter_diagnosis.EncounterID = dbo_encounter.id

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ICD10_Diagnosis_DE
  ON dbo_ICD10_Diagnosis_DE.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'ICD10'

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ICD9_Diagnosis_DE
  ON dbo_ICD9_Diagnosis_DE.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'ICD9'

LEFT JOIN problem_diagnosis
  ON problem_diagnosis.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'PROBLEM'

LEFT JOIN standard_concept_mapping
  ON dbo_encounter_diagnosis.DiagnosisType IN ('ICD9','ICD10')
 AND standard_concept_mapping.source_vocabulary_id = CASE
      WHEN dbo_encounter_diagnosis.DiagnosisType = 'ICD9'  THEN 'ICD9CM'
      WHEN dbo_encounter_diagnosis.DiagnosisType = 'ICD10' THEN 'ICD10CM'
    END
 AND standard_concept_mapping.source_concept_code = COALESCE(
      NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD9_Diagnosis_DE.EntryCode,  '[\\s\\u00A0]+', '')), ''),
      NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD10_Diagnosis_DE.EntryCode, '[\\s\\u00A0]+', '')), '')
    )

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_person
  ON dbo_person.id = dbo_encounter.patientid

LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE

WHERE 1=1
LIMIT 10;

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.condition_occurrence AS t
USING silver_condition_occurrence AS s
ON t.condition_occurrence_source_value = s.condition_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.condition_concept_id <=> s.condition_concept_id)
  OR NOT (t.condition_start_date <=> s.condition_start_date)
  OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
  OR NOT (t.condition_end_date <=> s.condition_end_date)
  OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
  OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
  OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.condition_source_value <=> s.condition_source_value)
  OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
  OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.condition_concept_id          = s.condition_concept_id,
  t.condition_start_date          = s.condition_start_date,
  t.condition_start_datetime      = s.condition_start_datetime,
  t.condition_end_date            = s.condition_end_date,
  t.condition_end_datetime        = s.condition_end_datetime,
  t.condition_type_concept_id     = s.condition_type_concept_id,
  t.condition_status_concept_id   = s.condition_status_concept_id,
  t.stop_reason                   = s.stop_reason,
  t.condition_source_value        = s.condition_source_value,
  t.condition_source_concept_id   = s.condition_source_concept_id,
  t.condition_status_source_value = s.condition_status_source_value,
  t.person_source_value           = s.person_source_value,
  t.provider_source_value         = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value     = s.visit_detail_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  condition_occurrence_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.condition_concept_id,
  s.condition_start_date,
  s.condition_start_datetime,
  s.condition_end_date,
  s.condition_end_datetime,
  s.condition_type_concept_id,
  s.condition_status_concept_id,
  s.stop_reason,
  s.condition_source_value,
  s.condition_source_concept_id,
  s.condition_status_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.condition_occurrence_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
%sql
-- Verify silver layer
SELECT * FROM _exponent.omop_silver.condition_occurrence LIMIT 10

In [0]:
%sql
-- Insert new mappings to source_to_condition_occurrence
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.condition_occurrence_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, condition_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.condition_occurrence
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
  ON s.condition_occurrence_source_value = x.condition_occurrence_source_value;

In [0]:
%sql
-- Verify mapping table
SELECT * FROM _exponent.omop_mapping.source_to_condition_occurrence LIMIT 10

In [0]:
%sql
-- Merge to Gold layer
-- Note: provider_id and visit_occurrence_id are NULL (mapping tables not created)
MERGE INTO _exponent.omop.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    stp.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    NULL AS provider_id,  -- source_to_provider not yet created
    NULL AS visit_occurrence_id,  -- source_to_visit_occurrence not yet created
    NULL AS visit_detail_id,  -- source_to_visit_detail not yet created
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
)
VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);

In [0]:
%sql
-- Verify gold layer
SELECT * FROM _exponent.omop.condition_occurrence LIMIT 10

In [0]:
%sql
-- Validation: Count records at each layer
SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.condition_occurrence
UNION ALL
SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_condition_occurrence
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.condition_occurrence

In [0]:
%sql
-- Data Quality: Check unmapped conditions (concept_id = 0)
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN condition_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_count,
  ROUND(100.0 * SUM(CASE WHEN condition_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_pct
FROM _exponent.omop.condition_occurrence